In [ ]:
from langgraph.graph import StateGraph
from typing import TypedDict, Dict, Any

In [ ]:
# Define state

class TeaState(TypedDict):
    teabag: int
    sugar: int
    jaggery: int
    milk: str
    tea: str
    milky: str
    mix: str
    final: str
    sweet:bool
    attempts: int

#nodes
def tea_bag(state: Dict[str, Any]) -> Dict[str, Any]:
    state["tea"] = f"Tea from {state['teabag']} teabags"
    return state

def milk(state: Dict[str, Any]) -> Dict[str, Any]:
    state["milky"] = f"{state['tea']}+{state['milk']} milk"
    return state

def add_sugar(state: Dict[str, Any]) -> Dict[str, Any]:
    state["sugar"] += 1
    state["attempts"] += 1
    state["mix"]= f" {state['milky']}+{state['sugar']} spoons sugar"
    return state

def add_jaggery(state: Dict[str, Any]) -> Dict[str, Any]:
    state["jaggery"] += 1
    state["attempts"] += 1
    state["mix"]= f" {state['milky']}+{state['jaggery']} cubes jaggery"
    return state

def taste_and_check(state: Dict[str, Any]) -> Dict[str, Any]:
    state["sweet"] = (state["sugar"] >= 2 or state["jaggery"] >= 2)
    return state

def serve(state: Dict[str, Any]) -> Dict[str, Any]:
    state["final"] = f"{state['mix']} → Served!"
    return state

#Decision function

def choose_sweetener(state: TeaState):
    if state["sugar"] > 0:
        return "add_sugar"
    return "add_jaggery"


def check_sweetness(state: TeaState):
    if state["sweet"]:
        return "serve"
    return "milk"

# Build graph
builder = StateGraph(TeaState)
builder.add_node("tea_bag", tea_bag)
builder.add_node("milk", milk)
builder.add_node("add_sugar", add_sugar)
builder.add_node("add_jaggery", add_jaggery)
builder.add_node("taste_and_check", taste_and_check)
builder.add_node("serve", serve)

builder.set_entry_point("tea_bag")
builder.add_edge("tea_bag", "milk")
builder.add_conditional_edges("milk", choose_sweetener,{
    "add_sugar":"add_sugar",
    "add_jaggery": "add_jaggery"})
builder.add_edge("add_sugar", "taste_and_check")
builder.add_edge("add_jaggery", "taste_and_check")
builder.add_conditional_edges("taste_and_check", check_sweetness, {
    "serve": "serve",
    "milk":"milk"})
builder.set_finish_point("serve")

from IPython.display import Image, display
graph = builder.compile()
# Visualize
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization failed:", e)

In [ ]:
initial_state = {
    "teabag": 1,
    "sugar": 0,
    "jaggery": 1,
    "milk": "1 cup milk",
    "tea": "",
    "milky": "",
    "mix": "",
    "final": "",
    "sweet": False,
    "attempts": 0,
    }

result = graph.invoke(initial_state)

import pandas as pd
result_df = pd.DataFrame([result])
result_df